# Creating a New Benchmark

## CircuitBench Tutorial 9

This notebook demonstrates how to create, validate, package, and register a new benchmark in CircuitBench. The workflow follows reproducible research practices and produces standardized benchmark metadata suitable for publication and reuse.

### Learning Objectives

- Create a new benchmark
- Define inputs and targets
- Register benchmark metadata
- Validate benchmark integrity
- Generate benchmark configuration files
- Export a reusable benchmark package


## Scientific Background

A benchmark should provide a reproducible evaluation task with standardized inputs, outputs, metadata, evaluation metrics, and documentation. CircuitBench ensures every benchmark follows a consistent schema to facilitate fair comparison across machine learning models.

In [ ]:
from pathlib import Path
import json
import warnings
import random

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
from circuitbench.benchmark import BenchmarkBuilder

## Initialize Benchmark Builder

In [ ]:
builder = BenchmarkBuilder()

## Load Source Dataset

In [ ]:
dataset = pd.read_csv("datasets/My_New_Dataset.csv")

dataset.head()

## Define Benchmark Metadata

In [ ]:
benchmark_metadata = {
    "name": "My_New_Benchmark",
    "version": "1.0.0",
    "description": "Example benchmark for CircuitBench.",
    "task": "Regression",
    "license": "Apache-2.0",
    "author": "CircuitBench Project",
    "random_seed": SEED,
}

benchmark_metadata

## Define Features and Target

In [ ]:
target_column = "target"

feature_columns = [c for c in dataset.columns if c != target_column]

len(feature_columns)

## Create Benchmark Object

In [ ]:
benchmark = builder.create(
    dataframe=dataset,
    features=feature_columns,
    target=target_column,
    metadata=benchmark_metadata,
)

benchmark

## Validate Benchmark

In [ ]:
validation_report = builder.validate(benchmark)

validation_report

## Benchmark Summary

In [ ]:
summary = {
    "Samples": dataset.shape[0],
    "Features": len(feature_columns),
    "Task": benchmark_metadata["task"],
    "Target": target_column,
    "Version": benchmark_metadata["version"],
}

pd.Series(summary)

## Register Benchmark

In [ ]:
builder.register(benchmark)

## List Available Benchmarks

In [ ]:
builder.list_benchmarks()

## Generate Benchmark Configuration

In [ ]:
configuration = {
    "benchmark_name": benchmark_metadata["name"],
    "dataset": "My_New_Dataset.csv",
    "task": benchmark_metadata["task"],
    "target": target_column,
    "features": feature_columns,
    "seed": SEED,
}

configuration

## Save Configuration

In [ ]:
output_dir = Path("new_benchmark")
output_dir.mkdir(exist_ok=True)

with open(output_dir / "benchmark_config.json", "w") as f:
    json.dump(configuration, f, indent=4)

## Save Metadata

In [ ]:
with open(output_dir / "metadata.json", "w") as f:
    json.dump(benchmark_metadata, f, indent=4)

## Create Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = dataset[feature_columns]
y = dataset[target_column]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, shuffle=True
)

print(f"Training Samples: {len(X_train)}")
print(f"Testing Samples: {len(X_test)}")

## Benchmark Quality Assessment

In [ ]:
quality_report = pd.DataFrame(
    {
        "Criterion": [
            "No Missing Values",
            "No Duplicate Rows",
            "Target Present",
            "At Least 100 Samples",
            "At Least 2 Features",
        ],
        "Passed": [
            dataset.isnull().sum().sum() == 0,
            dataset.duplicated().sum() == 0,
            target_column in dataset.columns,
            dataset.shape[0] >= 100,
            len(feature_columns) >= 2,
        ],
    }
)

quality_report

## Feature Summary

In [ ]:
feature_summary = pd.DataFrame(
    {
        "Feature": feature_columns,
        "Data Type": X.dtypes.astype(str),
        "Missing Values": X.isnull().sum().values,
        "Unique Values": X.nunique().values,
    }
)

feature_summary.head()

## Export Benchmark Files

In [ ]:
quality_report.to_csv(output_dir / "quality_report.csv", index=False)

feature_summary.to_csv(output_dir / "feature_summary.csv", index=False)

dataset.to_csv(output_dir / "benchmark_dataset.csv", index=False)

## Generate README

In [ ]:
readme = f"""# {benchmark_metadata['name']}

Version: {benchmark_metadata['version']}

Task: {benchmark_metadata['task']}

Samples: {dataset.shape[0]}

Features: {len(feature_columns)}

Target: {target_column}
"""

with open(output_dir / "README.md", "w") as f:
    f.write(readme)

## Benchmark Package Summary

In [ ]:
package_summary = {
    "benchmark": benchmark_metadata["name"],
    "version": benchmark_metadata["version"],
    "registered": True,
    "files_generated": [
        "benchmark_config.json",
        "metadata.json",
        "benchmark_dataset.csv",
        "quality_report.csv",
        "feature_summary.csv",
        "README.md",
    ],
}

package_summary

## Reproducibility Checklist

- Benchmark metadata fully documented.
- Dataset source recorded.
- Feature definitions preserved.
- Target variable specified.
- Random seed fixed.
- Quality checks completed.
- Configuration files exported.
- README generated.
- Benchmark package archived.


## Recommended Benchmark Package

```
new_benchmark/
├── benchmark_dataset.csv
├── benchmark_config.json
├── metadata.json
├── README.md
├── LICENSE
├── CITATION.cff
├── quality_report.csv
├── feature_summary.csv
└── benchmark_card.md
```


## Best Practices

1. Use clearly documented input and output variables.
2. Include units for every numerical feature.
3. Keep benchmark versions immutable after publication.
4. Validate benchmark integrity before release.
5. Provide reproducible train/test splits.
6. Publish metadata together with the dataset.
7. Archive benchmark releases using persistent identifiers where possible.


# Summary

In this notebook we:

- Created a new CircuitBench benchmark.
- Defined benchmark metadata and configuration.
- Registered the benchmark.
- Performed automated validation and quality assessment.
- Generated benchmark documentation.
- Exported a reusable benchmark package.

Following this workflow ensures that benchmarks are standardized, reproducible, and immediately compatible with the CircuitBench benchmarking framework.